# Project 1

Welcome to the first NLP1 project. The goal of the first tutorial was to get you familiar with text segmentation and simple text classification for topic detection.
Given the skills you have developed in the first week, you will now be applying them within a different dataset and task.

We are recommending a set of tasks and associated datasets; using those, you will be experimenting on problems introduced during the tutorials. You will be applying hands-on the experiments you have already explored in the last tutorial, and taking a deeper dive: extending your experiments, trying different set ups, possibly considering different evaluation techniques and analysing errors. 

The week after the deadline, your TA will perform a short oral assessment of the skills and knowledge you acquired (make sure to check the official schedule for this), probing you to answer questions about what you learnt, about what you did and about why you did the things you did (as opposed to some reasonable alternatives) as well as pros and cons of your method, and so on. 

As the tutorials will be building upon your knowledge throughout the weeks, the task and dataset you choose now will be the ones you use in projects 2 and 3; and the ones you will be finally writing about in your final blog post.  

## Get to work!

The next section has a checklist to help you not lose track of what needs to be done. Here we have a more informal guide, with prompts and questions for you. 


1. [Choose a dataset](http://nlp-illc.github.io/nlp1/). Spend some time understanding how the dataset looks and some statistics of it (e.g. how many instances of each classification label we have?)
2. Remember what we learnt during the tutorial: to develop any classifier of the kind we saw, we needed a way to segment text. Do you want to use the tokeniser we have developed? Do you need to train a new one? If you do, what hyperparameters shall you use? What data shall you use to train it? Consider that your dataset may be too small to train a tokeniser! Think whether you will need a larger resource of similar nature to train a tokeniser.
3. Start thinking how to represent your documents into features. We have experimented with token occurences and frequencies. Can you think of any other ways, possibly? 
4. We had experimented with Logistic Regression and a shallow neural network. You can experiment with and assess different architectures now and their hyper parameters (e.g. add layers, possibly change the activation functions, drop out rates).
5. Evaluate your models! Was the evaluation method adequate for your new experimental setup? Do you want to assess other aspects of performance?
6. Analyse the various models you assessed and try to decide on which is the best.

Be mindful that you will be using your experiments to write up your final blog, so make sure they are clean and neat!

You can write your own code and you can reuse code from the tutorial and its packages or from other sources. You may use coding assistants if you like, but use it mindfully as you take full responsibility for this kind of (semi-)automated coding. Also, keep a record of the kinds of things you delegate to any form of assistant, as you will be asked to disclose this in your blog post (not because we will pass a value judgement on it, but because it is a widespread and good practice to document such things).

You may use jupyter notebooks or python files, it's up to you as we will not directly assess your code. But, once again, be organised, as we will ask you to share a reproducible version of your experiments along with your blog post (again, because this is how real research is publicised). 

If you are unsure about anything, ask your TA! They are always happy to help you! Happy coding! :)

## Checklist

What **we expect you to do**:

- [ ] Compare different systems of text classification for your choice of data/task.
- Your comparisons should cover:
    - [ ] the two representation spaces in the tutorial (token indicators and bag-of-words) and an additional one of your choice.
    - [ ] the two kinds of architectures in the tutorial (log-linear and single-hidden-layer MLP) and an additional one of your choice (such as a modification of the MLP).
- [ ] Your comparisons must be based on cross-validation and a relevant metric (accuracy or else).
- [ ] You should make use of Bayesian estimation to help you analyse results in relation to a region of practical equivalence (ROPE), and you can motivate the ROPE size yourself.

What you need to **bring to the oral assessment**:
* You will be asked questions about the points above, you can bring a slide deck with up to five slides (one for each of the following topics _data and metrics_, _features_, _architectures_, _results_, _extra_); you _will not_ give a presentation, rather, these slides are to be thought of as "backup slides" that you can use to guide your answers to the questions the TA will ask you.
* You may also be asked questions about the tutorial, for those, you don't need to prepare any deliverable (a good way to prepare is to work through the tutorial and reflect on the questions there).

What you should **keep in mind** given that you will eventually write a blog post about your work throughout the course:
* Organise your code;
* Summarise your motivations and findings;
* Document your sources and any form of AI-assistance you use.

## LOADS DATA

In [13]:
import os
import kagglehub

path = kagglehub.dataset_download("debarshichanda/goemotions")
print(path)
print(os.listdir(path))

/Users/ioanadumitrache/.cache/kagglehub/datasets/debarshichanda/goemotions/versions/6
['replace_emotions.py', 'tables', 'calculate_metrics.py', 'extract_words.py', 'README.md', 'analyze_data.py', 'GoEmotionsFormat.PNG', 'data', 'goemotions_model_card.pdf', 'plots']


## READS DATA

In [14]:
import pandas as pd

train_df = pd.read_csv(os.path.join(path, "data/train.tsv"), sep="\t", header=None, names=["text", "#","labels"])

## TOKENIZER

In [15]:
import tokenizer
train_df["text"] = train_df["text"].str.lower()
train_df = tokenizer.tokenize_text(train_df)
train_df = train_df.drop(columns=["#", "text"])
print(train_df.head())

    labels                                        text_tokens
0  eebbqej  [my, favourite, food, is, anything, i, did, n'...
1  ed00q6i  [now, if, he, does, off, himself, ,, everyone,...
2  eezlygj              [why, the, fuck, is, bayless, isoing]
3  ed7ypvh                  [to, make, her, feel, threatened]
4  ed0bdzj                         [dirty, southern, wankers]


## REPRESENTATION SPACE

In [16]:
import representation_spaces

vocab = representation_spaces.vocab(train_df)
# print(vocab)
print(len(vocab))
train_df["token_indicators"] = representation_spaces.token_indicators(train_df["text_tokens"], vocab)
print(train_df["token_indicators"].head())
print(train_df["token_indicators"][0].toarray()[0][:20])   # first 20 values of doc 0

train_df["bag_of_words"] = representation_spaces.bag_of_words(train_df["text_tokens"], vocab)

28848
0    <Compressed Sparse Row sparse matrix of dtype ...
1    <Compressed Sparse Row sparse matrix of dtype ...
2    <Compressed Sparse Row sparse matrix of dtype ...
3    <Compressed Sparse Row sparse matrix of dtype ...
4    <Compressed Sparse Row sparse matrix of dtype ...
Name: token_indicators, dtype: object
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]


In [17]:
#print(len(vocab[train_df["token_indicators"][0]]))
print(train_df.head())
print("########")
print(train_df["token_indicators"][0].toarray()[0][104])

    labels                                        text_tokens  \
0  eebbqej  [my, favourite, food, is, anything, i, did, n'...   
1  ed00q6i  [now, if, he, does, off, himself, ,, everyone,...   
2  eezlygj              [why, the, fuck, is, bayless, isoing]   
3  ed7ypvh                  [to, make, her, feel, threatened]   
4  ed0bdzj                         [dirty, southern, wankers]   

                                    token_indicators  \
0  <Compressed Sparse Row sparse matrix of dtype ...   
1  <Compressed Sparse Row sparse matrix of dtype ...   
2  <Compressed Sparse Row sparse matrix of dtype ...   
3  <Compressed Sparse Row sparse matrix of dtype ...   
4  <Compressed Sparse Row sparse matrix of dtype ...   

                                        bag_of_words  
0  <Compressed Sparse Row sparse matrix of dtype ...  
1  <Compressed Sparse Row sparse matrix of dtype ...  
2  <Compressed Sparse Row sparse matrix of dtype ...  
3  <Compressed Sparse Row sparse matrix of dtype ...

In [18]:
from scipy.sparse import vstack

X_ind = vstack(train_df["token_indicators"])
X_bow = vstack(train_df["bag_of_words"])
y_train = train_df["labels"]
print(X_ind.shape, X_bow.shape)

(43410, 28848) (43410, 28848)


In [ ]:
import architecture

# token indicator
mlp = architecture.build_mlp()
mlp.fit(X_ind, y_train)
print("dev accuracy:", mlp.score(X_dev_ind, dev_df["labels"]))

# bag-of-words
mlp_bow = architecture.build_mlp()
mlp_bow.fit(X_bow, y_train)
print("bag-of-words + MLP, dev accuracy:", mlp_bow.score(X_dev_bow, dev_df["labels"]))